# Clinic Visits — Data Cleaning Notebook

Pipeline: **Raw data → Profile → Parse dates → Standardise categories → Resolve duplicates →
Handle missing values → Handle impossible values → Resolve orphan keys → Validate → Write cleaned tables**

The raw file is never modified — every step below reads from `raw` (or an intermediate
DataFrame) and writes a new, clearly-named output. See `data_quality_notes.md` for the
narrative write-up of every finding and treatment decision this notebook implements.

In [ ]:
import pandas as pd
import numpy as np

RAW_PATH = "data/raw_clinic_visits.csv"   # exported once from Clinic_Visits.xlsx, kept untouched
pd.set_option("display.max_columns", None)

raw = pd.read_csv(RAW_PATH)
print(raw.shape)
raw.head()

## 1. Profile the raw data

Check shape, dtypes, and an initial pass at nulls/uniques before touching anything.

In [ ]:
print(raw.dtypes)
print()
print(raw.isnull().sum())
print()
print("Rows:", len(raw))
print("Unique visit_id:", raw['visit_id'].nunique())
print("Unique visit_type raw values:", raw['visit_type'].unique())

**Finding:** no column reports a pandas `NaN`, but `visit_id` has fewer unique values
than rows (duplicates), and `visit_type` contains a literal text `'[NULL]'` rather than a
true null — a classic case of missingness that a blind `.isnull()` check misses.

## 2. Resolve duplicate visits

Check whether duplicated `visit_id`s are full-row duplicates before deciding how to treat them.

In [ ]:
dup_mask = raw.duplicated(subset="visit_id", keep=False)
duplicate_log = raw[dup_mask].sort_values("visit_id")
print("Rows involved in a visit_id duplicate:", len(duplicate_log))

# Are they full-row duplicates (ignoring the spreadsheet row number '#')?
full_dupe_check = duplicate_log.drop(columns=["#"]).duplicated(keep=False)
print("Of those, full-row duplicates (excluding row number):", full_dupe_check.sum())

duplicate_log.to_csv("data/audit_duplicate_visit_id.csv", index=False)
duplicate_log.head(6)

In [ ]:
# Every duplicated visit_id is an exact repeat -> safe to drop the repeat, not merge/investigate.
df = raw.drop_duplicates(subset="visit_id", keep="first").copy()
print("Rows after de-duplication:", len(df))
assert df['visit_id'].is_unique

## 3. Parse `visit_date`

Two formats are mixed in the same column: native Excel datetimes and `DD/MM/YYYY` text.

In [ ]:
print(df['visit_date'].apply(type).value_counts())

# The raw CSV export stores every value as text, in one of two formats:
#   "DD/MM/YYYY"                 (the 963 rows that were already text in the source workbook)
#   "YYYY-MM-DD HH:MM:SS"        (every other row, once a native Excel datetime hits a CSV)
# Try the day-first format first (the one that is genuinely ambiguous), then fall back.
def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    x = str(x).strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        parsed = pd.to_datetime(x, format=fmt, errors="coerce")
        if pd.notna(parsed):
            return parsed
    return pd.to_datetime(x, errors="coerce")  # last resort, logged as unparsed if it fails

df["visit_date"] = df["visit_date"].apply(parse_date)

bad_dates = df[df["visit_date"].isna()]
print("Rows that failed to parse:", len(bad_dates))
print("Date range:", df["visit_date"].min(), "to", df["visit_date"].max())
bad_dates.to_csv("data/audit_unparseable_dates.csv", index=False)

**Finding:** 0 rows failed to parse once the text format is read explicitly as
`DD/MM/YYYY`. Range is 1 Jan 2024 – 30 Dec 2025, with no impossible (future or absurdly
old) dates. This sizes `DimDate`.

## 4. Standardise `visit_type`

Trim whitespace, fold case, and map every spelling to one of three real categories.
The literal `'[NULL]'` values become their own explicit `Unknown` category — they are
**not** guessed into Outpatient/Follow-up/Emergency.

In [ ]:
df["visit_type_raw"] = df["visit_type"]
normalised = df["visit_type"].astype(str).str.strip().str.upper()

mapping = {
    "OUTPATIENT": "Outpatient",
    "FOLLOW-UP":  "Follow-up",
    "EMERGENCY":  "Emergency",
    "[NULL]":     "Unknown",
}
df["visit_type"] = normalised.map(mapping)

unmapped = df[df["visit_type"].isna()]
if len(unmapped):
    print("UNMAPPED VALUES — investigate before proceeding:", unmapped["visit_type_raw"].unique())
else:
    print("All raw spellings mapped successfully.")

df["visit_type"].value_counts(dropna=False)

## 5. Flag impossible `wait_minutes`

A wait time cannot be negative. Flag rather than delete — the visit itself is still real.

In [ ]:
df["is_valid_wait"] = df["wait_minutes"] >= 0
neg_wait_log = df[~df["is_valid_wait"]]
print("Negative wait_minutes rows:", len(neg_wait_log))
print(neg_wait_log["wait_minutes"].describe())
neg_wait_log.to_csv("data/audit_negative_wait_minutes.csv", index=False)

## 6. Resolve orphan / out-of-range keys

No separate patient/doctor/department master file was supplied with this dataset, so a
true foreign-key anti-join isn't possible. This section does the closest available
check: range plausibility against the scope described for the project (900 patients,
40 doctors, 8 departments).

In [ ]:
dept_ok = df["department_id"].between(1, 8)
doc_ok  = df["doctor_id"].between(1, 40)
print("department_id out of range 1-8:", (~dept_ok).sum())
print("doctor_id out of range 1-40:", (~doc_ok).sum())

df["is_registered_patient"] = df["patient_id"] < 10000
walkins = df[~df["is_registered_patient"]]
print("patient_id outside 1-900-ish registered range:", len(walkins))
print(sorted(walkins["patient_id"].unique()))
walkins.to_csv("data/audit_unregistered_patient_ids.csv", index=False)

**Finding:** every `department_id` and `doctor_id` falls inside its expected range —
no orphans by this proxy check. 62 rows have `patient_id` in a distinct 90000–90061
block, consistent with a walk-in/unregistered-patient placeholder range rather than
noise. Flagged, not deleted, pending confirmation from the patient-master owner.

## 7. Validate the cleaned population

Reconcile every population size before building anything downstream.

In [ ]:
print("Raw rows in source file:                         25,100 (see data/raw_clinic_visits.csv)")
print(f"Unique visits after de-duplication:               {len(df):,}")
print(f"...with a valid (non-negative) wait time:         {df['is_valid_wait'].sum():,}")
print(f"...of those, with a known (non-Unknown) visit type: {(df['is_valid_wait'] & (df['visit_type']!='Unknown')).sum():,}")

## 8. Headline analysis (cross-check figures against SQL and, later, Power BI)

In [ ]:
valid = df[df["is_valid_wait"]].copy()

print("Overall average wait (valid rows):", round(valid["wait_minutes"].mean(), 2))
print()

by_dept = valid.groupby("department_id")["wait_minutes"].mean().sort_values(ascending=False)
print("Average wait by department:")
print(by_dept.round(2))

In [ ]:
by_type = valid.groupby("visit_type")["wait_minutes"].mean().sort_values(ascending=False)
print("Average wait by visit type:")
print(by_type.round(2))
print()

pivot = valid.groupby(["department_id", "visit_type"])["wait_minutes"].mean().unstack().round(1)
pivot

In [ ]:
valid["month"] = valid["visit_date"].dt.to_period("M")
monthly = valid.groupby("month")["wait_minutes"].mean()
print("Highest month:", monthly.idxmax(), round(monthly.max(), 2))
print("Lowest month:", monthly.idxmin(), round(monthly.min(), 2))
monthly.round(1)

## 9. Build the star schema tables and write cleaned outputs

`DimPatient`, `DimDoctor` and `DimDepartment` are **surrogate** dimensions built from the
distinct keys present in the fact table, because no separate master files were supplied.
Attributes the brief describes (BirthYear, Sex, Region, doctor Name, YearsOfExperience,
department Floor) are left out rather than invented.

In [ ]:
# DimDate — every calendar day in range, so trend charts don't silently skip empty periods
all_dates = pd.date_range(df["visit_date"].min(), df["visit_date"].max(), freq="D")
dim_date = pd.DataFrame({"Date": all_dates})
dim_date["DateKey"]   = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Day"]       = dim_date["Date"].dt.day
dim_date["Week"]      = dim_date["Date"].dt.isocalendar().week
dim_date["Month"]     = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.strftime("%B")
dim_date["Quarter"]   = dim_date["Date"].dt.quarter
dim_date["Year"]      = dim_date["Date"].dt.year
dim_date = dim_date[["DateKey","Date","Day","Week","Month","MonthName","Quarter","Year"]]
dim_date.to_csv("data/dim_date.csv", index=False)
dim_date.head()

In [ ]:
# DimDepartment / DimDoctor — surrogate labels only
dim_department = pd.DataFrame({"DepartmentKey": sorted(df["department_id"].unique())})
dim_department["DepartmentName"] = "Department " + dim_department["DepartmentKey"].astype(str)
dim_department.to_csv("data/dim_department.csv", index=False)

dim_doctor = pd.DataFrame({"DoctorKey": sorted(df["doctor_id"].unique())})
dim_doctor["DoctorLabel"] = "Doctor " + dim_doctor["DoctorKey"].astype(str)
dim_doctor.to_csv("data/dim_doctor.csv", index=False)

print(dim_department.shape, dim_doctor.shape)

In [ ]:
# DimVisitType — 'Unknown' is a real 4th member, not an error
dim_visittype = pd.DataFrame({
    "VisitTypeKey": [1, 2, 3, 4],
    "VisitType": ["Outpatient", "Follow-up", "Emergency", "Unknown"],
})
dim_visittype.to_csv("data/dim_visittype.csv", index=False)
vt_map = dict(zip(dim_visittype["VisitType"], dim_visittype["VisitTypeKey"]))
dim_visittype

In [ ]:
# DimPatient — surrogate, flags registration plausibility
dim_patient = (
    df[["patient_id", "is_registered_patient"]]
    .drop_duplicates("patient_id")
    .rename(columns={"patient_id": "PatientKey", "is_registered_patient": "IsRegisteredPatient"})
)
dim_patient.to_csv("data/dim_patient.csv", index=False)
print(dim_patient.shape)

In [ ]:
# FactVisits — grain: one row per cleaned, de-duplicated visit
fact = df.copy()
fact["DateKey"] = fact["visit_date"].dt.strftime("%Y%m%d").astype(int)
fact["VisitTypeKey"] = fact["visit_type"].map(vt_map)

fact_visits = fact.rename(columns={
    "visit_id": "VisitID",
    "patient_id": "PatientKey",
    "doctor_id": "DoctorKey",
    "department_id": "DepartmentKey",
    "wait_minutes": "WaitMinutes",
    "consultation_fee": "Fee",
    "is_valid_wait": "IsValidWait",
    "is_registered_patient": "IsRegisteredPatient",
})[[
    "VisitID", "DateKey", "PatientKey", "DoctorKey", "DepartmentKey", "VisitTypeKey",
    "WaitMinutes", "Fee", "IsValidWait", "IsRegisteredPatient",
]]

fact_visits.to_csv("data/fact_visits.csv", index=False)
print(fact_visits.shape)
fact_visits.head()

## 10. Final validation before handing off to Power BI

Confirm the cleaned outputs reconcile to the numbers documented in
`data_quality_notes.md` before importing anything into the model.

In [ ]:
assert fact_visits["VisitID"].is_unique
assert len(fact_visits) == 25000
assert fact_visits["IsValidWait"].sum() == 24950
assert fact_visits.merge(dim_date, left_on="DateKey", right_on="DateKey", how="left")["Date"].isna().sum() == 0
assert fact_visits["DepartmentKey"].isin(dim_department["DepartmentKey"]).all()
assert fact_visits["DoctorKey"].isin(dim_doctor["DoctorKey"]).all()
assert fact_visits["VisitTypeKey"].isin(dim_visittype["VisitTypeKey"]).all()
assert fact_visits["PatientKey"].isin(dim_patient["PatientKey"]).all()
print("All referential-integrity and row-count checks passed.")
print("Ready for Power BI import: fact_visits.csv + dim_date.csv + dim_department.csv "
      "+ dim_doctor.csv + dim_visittype.csv + dim_patient.csv")